# Progressive Transformers for USL Sign Language Production

Trains a Progressive Transformer (Saunders et al., ECCV 2020) on Ukrainian Sign Language data.

**Pre-requisites:**
1. Run `02_prepare_progressive_transformers.ipynb` (extracts poses, prepares data, uploads `usl-pt-training-ready.zip` to Google Drive)

**Requirements:** GPU runtime (T4 or better).

## 1. Setup

Mounts Google Drive and extracts `usl-pt-training-ready.zip`.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/ucu-text-to-sign")
archive = DRIVE_DIR / "usl-pt-training-ready.zip"
assert archive.exists(), f"Archive not found: {archive}\nRun notebook 02 first to create and upload usl-pt-training-ready.zip"

SLP_ROOT = "/content/progressive_transformers"
!mkdir -p {SLP_ROOT}
!unzip -q -o {archive} -d {SLP_ROOT}/

print(f"Extracted {archive.name} to {SLP_ROOT}")

In [ ]:
import os, sys

# Verify structure
!echo "=== Config ==="
!ls {SLP_ROOT}/Configs/
!echo "=== Data ==="
!ls {SLP_ROOT}/Data/usl/
!echo "=== Python files ==="
!ls {SLP_ROOT}/*.py | head -20

!pip install -q pyyaml scipy tensorboard

## 2. Verify Data

In [ ]:
data_dir = Path(SLP_ROOT) / 'Data/usl'
for split in ['train', 'dev', 'test']:
    for ext in ['text', 'skels', 'files']:
        path = data_dir / f'{split}.{ext}'
        if path.exists():
            with open(path) as f:
                n = sum(1 for _ in f)
            print(f'{split}.{ext}: {n} lines')
        else:
            print(f'{split}.{ext}: MISSING!')

vocab_path = Path(SLP_ROOT) / 'Configs/usl_src_vocab.txt'
if vocab_path.exists():
    with open(vocab_path) as f:
        vocab = f.readlines()
    print(f'\nVocabulary: {len(vocab)} tokens')
    print(f'First 10: {[t.strip() for t in vocab[:10]]}')
else:
    print('\nVocabulary file MISSING!')

## 3. Sanity Check: Load Data & Build Model

In [ ]:
os.chdir(SLP_ROOT)
sys.path.insert(0, SLP_ROOT)

from helpers import load_config
from data import load_data, make_data_iter

cfg = load_config('./Configs/USL.yaml')
train_data, dev_data, test_data, src_vocab, trg_vocab = load_data(cfg)

print(f'Train: {len(train_data)} sequences')
print(f'Dev:   {len(dev_data)} sequences')
print(f'Test:  {len(test_data)} sequences')
print(f'Vocab: {len(src_vocab)} tokens')
print(f'Target size: {len(trg_vocab)}')

# Check one batch
train_iter = make_data_iter(train_data, batch_size=4, train=True, shuffle=True)
batch = next(iter(train_iter))
src, src_lengths = batch.src
print(f'\nBatch src shape:  {src.shape}')
print(f'Batch trg shape:  {batch.trg.shape}')
print(f'Src lengths:      {src_lengths.tolist()}')

## 4. Train

In [ ]:
os.chdir(SLP_ROOT)
!python __main__.py train ./Configs/USL.yaml

## 5. Monitor Training (TensorBoard)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {SLP_ROOT}/Models/USL/tensorboard/

## 6. Evaluate & Visualise

In [ ]:
os.chdir(SLP_ROOT)

from training import test as run_test
from helpers import get_latest_checkpoint

ckpt = get_latest_checkpoint('./Models/USL', post_fix='_best')
if ckpt:
    print(f'Best checkpoint: {ckpt}')
    run_test(cfg_file='./Configs/USL.yaml', ckpt=ckpt)
else:
    print('No best checkpoint found yet.')

# --- Evaluate on TRAIN set (replication sanity check) ---
# With a small dataset, verifying the model can reproduce training examples
# is an important sanity check.
if ckpt:
    print('\n=== Evaluating on TRAIN set ===')
    import training as training_module
    # Monkey-patch to use train data instead of test
    _orig_test = training_module.test
    def test_on_train(cfg_file, ckpt):
        from helpers import load_config, load_checkpoint
        from model import build_model
        from data import load_data
        from prediction import validate_on_data
        from training import TrainManager

        cfg = load_config(cfg_file)
        model_dir = cfg["training"]["model_dir"]
        batch_size = cfg["training"].get("eval_batch_size", cfg["training"]["batch_size"])
        batch_type = cfg["training"].get("eval_batch_type", cfg["training"].get("batch_type", "sentence"))
        use_cuda = cfg["training"].get("use_cuda", False)
        eval_metric = cfg["training"]["eval_metric"]
        max_output_length = cfg["training"].get("max_output_length", None)

        train_data, dev_data, test_data, src_vocab, trg_vocab = load_data(cfg=cfg)
        model_checkpoint = load_checkpoint(ckpt, use_cuda=use_cuda)
        model = build_model(cfg, src_vocab=src_vocab, trg_vocab=trg_vocab)
        model.load_state_dict(model_checkpoint["model_state"])
        if use_cuda:
            model.cuda()

        trainer = TrainManager(model=model, config=cfg, test=True)

        score, loss, references, hypotheses, inputs, all_dtw_scores, file_paths = \
            validate_on_data(
                model=model, data=train_data, batch_size=batch_size,
                max_output_length=max_output_length, eval_metric=eval_metric,
                loss_function=None, batch_type=batch_type, type="train_inf"
            )
        print(f'Train DTW score: {score:.4f}')

        display = list(range(min(len(hypotheses), 10)))  # first 10
        trainer.produce_validation_video(
            output_joints=hypotheses, inputs=inputs, references=references,
            model_dir=model_dir, display=display, type="train_replication",
            file_paths=file_paths,
        )

    test_on_train(cfg_file='./Configs/USL.yaml', ckpt=ckpt)

In [ ]:
# Display a generated video
from IPython.display import HTML
from base64 import b64encode

test_videos_dir = Path(SLP_ROOT) / 'Models/USL/test_videos'
if test_videos_dir.exists():
    videos = sorted(test_videos_dir.glob('*.mp4'))
    if videos:
        video_path = videos[0]
        print(f'Showing: {video_path.name}')
        with open(video_path, 'rb') as f:
            mp4 = b64encode(f.read()).decode()
        display(HTML(f'<video controls width=650><source src="data:video/mp4;base64,{mp4}" type="video/mp4"></video>'))
    else:
        print('No test videos generated yet.')
else:
    print('Test videos directory not found.')

In [ ]:
# Check validation report
val_report = Path(SLP_ROOT) / 'Models/USL/validations.txt'
if val_report.exists():
    with open(val_report) as f:
        lines = f.readlines()
    print(f'Validation report ({len(lines)} entries):')
    print('First 5:')
    for line in lines[:5]:
        print(f'  {line.strip()}')
    print('Last 5:')
    for line in lines[-5:]:
        print(f'  {line.strip()}')
else:
    print('No validation report found.')

In [ ]:
# Download model checkpoint
from google.colab import files as colab_files

ckpt_path = Path(SLP_ROOT) / 'Models/USL/best.ckpt'
if ckpt_path.exists():
    colab_files.download(str(ckpt_path))
else:
    print('No best checkpoint to download.')